In [1]:
from langchain_community.embeddings import DashScopeEmbeddings
from pymilvus import MilvusClient, AnnSearchRequest, WeightedRanker, RRFRanker

from app.core.config import settings

# 1.准备客户端
client = MilvusClient(uri=settings.rag.milvus_url)

# 2.准备向量模型
embedding = DashScopeEmbeddings(
    model="qwen3.7-text-embedding"
)

# 3.准备集合名称
collection_name = "collection_name_1"

# 4.定义混合检索的函数
def hybrid_search(query: str, ranker):
    # 4.1 稠密向量检索请求
    dense_search = AnnSearchRequest(
        data=[embedding.embed_query(query)],
        anns_field="dense",
        param={"metric_type": "COSINE"},
        limit=3
    )
    # 4.2 稀疏向量检索请求
    sparse_search = AnnSearchRequest(
        data=[query],
        anns_field="sparse",
        param={"metric_type": "BM25"},
        limit=3
    )
    # 4.3 组装请求列表
    request = [dense_search, sparse_search]
    # 4.4 执行正式的请求
    search_result = client.hybrid_search(
        collection_name, request,
        ranker, limit=3,
        output_fields=["h2", "content"]
    )
    # 4.5 测试查询结果
    for hits in search_result:
        for hit in hits:
            print("======================")
            print(hit)

C:\Users\75\AppData\Local\Temp\ipykernel_20500\3206045067.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import DashScopeEmbeddings


In [3]:
# 加权融合排序
ranker = WeightedRanker(0.6, 0.4)
# hybrid_search("教育的负面作用是什么?", ranker)
hybrid_search("给教师的建议是谁写的?", ranker)

{'id': 15, 'distance': 0.5019742250442505, 'entity': {'content': '# 第二章 教育基本原理  \n## 第一节 教育的功能  \n### （一）个体发展功能和社会发展功能  \n教育的正向功能（积极功能）指教育有助于社会进步和个体发展的积极影响和作用。  \n教育的负向功能（消极功能）指阻碍社会进步和个体发展的消极影响和作用。  \n教育的显性功能是指教育活动依照教育目的，在实际运行中所出现的与之相吻合的结果。  \n教育的隐性功能指伴随显性功能所出现的非预期性的功能。', 'h2': '第一节 教育的功能'}}
{'id': 18, 'distance': 0.4744347035884857, 'entity': {'content': '### （三）文化与教育的关系  \n文化对教育的影响：文化影响着教育的价值取向；文化影响着教育的课程内容；文化影响着育人模式；文化影响着教育教学方法的使用。  \n教育对文化的作用表现在：教育传递和保存文化；教育传播和交流文化；教育选择和提升文化；教育创造和更新文化。', 'h2': '第二节 教育和社会的关系'}}
{'id': 19, 'distance': 0.467794269323349, 'entity': {'content': '### （四）科技与教育的关系  \n科技对教育的作用主要表现在：科技改变着人们关于知识、学习和教育的观念；科技变革着现代教育的课程设置和内容体系；科技变革着现代教育手段、方式和方法。  \n教育对科技的作用主要表现在：教育传播科技；教育创新科技。', 'h2': '第二节 教育和社会的关系'}}


In [5]:
# RRF融合排序
ranker = RRFRanker(k=60)
# hybrid_search("教育的负面作用是什么?", ranker)
hybrid_search("给教师的建议是谁写的?", ranker)

{'id': 15, 'distance': 0.016393441706895828, 'entity': {'h2': '第一节 教育的功能', 'content': '# 第二章 教育基本原理  \n## 第一节 教育的功能  \n### （一）个体发展功能和社会发展功能  \n教育的正向功能（积极功能）指教育有助于社会进步和个体发展的积极影响和作用。  \n教育的负向功能（消极功能）指阻碍社会进步和个体发展的消极影响和作用。  \n教育的显性功能是指教育活动依照教育目的，在实际运行中所出现的与之相吻合的结果。  \n教育的隐性功能指伴随显性功能所出现的非预期性的功能。'}}
{'id': 18, 'distance': 0.016129031777381897, 'entity': {'h2': '第二节 教育和社会的关系', 'content': '### （三）文化与教育的关系  \n文化对教育的影响：文化影响着教育的价值取向；文化影响着教育的课程内容；文化影响着育人模式；文化影响着教育教学方法的使用。  \n教育对文化的作用表现在：教育传递和保存文化；教育传播和交流文化；教育选择和提升文化；教育创造和更新文化。'}}
{'id': 19, 'distance': 0.01587301678955555, 'entity': {'h2': '第二节 教育和社会的关系', 'content': '### （四）科技与教育的关系  \n科技对教育的作用主要表现在：科技改变着人们关于知识、学习和教育的观念；科技变革着现代教育的课程设置和内容体系；科技变革着现代教育手段、方式和方法。  \n教育对科技的作用主要表现在：教育传播科技；教育创新科技。'}}
